## **Cropland Area Estimation -- Bungoma & Muranga Counties, Kenya (2025 season)**
### **Large-Group Training Variant**

This notebook is a **single-round variant** of `area_estimation_bungoma2025.ipynb`, for training sessions with many participants. It reuses that notebook's map import/reprojection, sample export, and area/accuracy estimation steps unchanged, but replaces the pilot-sample-and-Neyman-allocation sample design with a simpler one sized directly to the training group: a fixed total sample quota (`n_participants * n_samples_per_participant`), split evenly and independently across counties.

Bungoma and Muranga are estimated **independently** -- own raster, own pixel counts, own allocation, own area/accuracy estimate -- but sampled and annotated together: one stratified random sample is drawn per county, then both are merged into a single shuffled, single-id sample for one combined annotation round in STAC Notator.

Author: Josef Wagner ([University of Strasbourg](https://www.unistra.fr/fr), [NASA Harvest](https://www.nasaharvest.org/)) jwagner@unistra.fr

### **Workflow**

1. Reproject each county's map to its own auto-derived equal-area projection.
2. Set strata and the shared sample-size parameters (participants x samples-per-participant).
3. Compute pixel counts, per county.
4. Set a per-county `min_allocation`, then allocate each county's share of the total sample proportionally across its own strata.
5. Draw a stratified random sample per county, merge them, shuffle once, assign one shared id space, and export a single combined CSV for annotation in STAC Notator (single **checkpoint**).
6. Load the annotated sample back in and split it by county (via a locally-kept id-to-county lookup, not a column trusted to survive the annotation tool's export).
7. Compute the design-based cropland area estimate, per county, with uncertainty.
8. Compute map accuracy (overall, user's, producer's), per county, with uncertainty.

How the exported sample's rows get distributed to individual participants is a logistics step outside this notebook.

See the **Background** section below (after Step 1) for a step-by-step recap of the statistically rigorous pilot + Neyman process this notebook simplifies, and why.

### **Setup**
First, install packages and import helper functions

In [ ]:
## The tricky part: installing gdal

# Update packages
!apt-get update -qq

# Install GDAL system libraries
!apt-get install -y gdal-bin libgdal-dev

import os
os.environ['CPLUS_INCLUDE_PATH'] = '/usr/include/gdal'
os.environ['C_INCLUDE_PATH'] = '/usr/include/gdal'

%pip install --upgrade pip -q
%pip install numpy pandas shapely fiona geopandas scikit-learn pyproj -q
# Optional: install GDAL Python bindings (match the system version)
%pip install gdal==$(gdal-config --version) -q

from osgeo import gdal, ogr, osr

print("GDAL version:", gdal.__version__)

In [ ]:
def in_colab():
  try:
    import google.colab
    return True
  except ImportError:
    return False

if in_colab():
  # git-lfs is required to fetch the real input_data/*.tif raster (LFS-tracked);
  # without it, the clone would only pull a small LFS pointer stub file.
  !apt-get install -y git-lfs -qq
  !git lfs install
  import os
  repo_dir = 'E2E_StacNotator_AreaEstimation_Training'
  repo_url = 'https://github.com/jowa-ea/E2E_StacNotator_AreaEstimation_Training.git'

  if os.path.basename(os.getcwd()) == repo_dir and os.path.isdir('.git'):
    # Re-running this cell in an already-warm runtime: a prior run's %cd
    # persists across cell re-executions within the same kernel, so we're
    # already sitting inside the clone -- checking for a same-named
    # subfolder (as opposed to checking whether we're already IN it) would
    # find nothing and clone again into ourselves, nesting the repo inside
    # itself. Just pull in place instead.
    !git pull
  elif os.path.isdir(repo_dir):
    # Cloned earlier in this runtime, but this cell hasn't cd'ed into it yet
    # (e.g. re-run after a kernel restart that kept the disk around).
    %cd {repo_dir}
    !git pull
  else:
    !git clone {repo_url}
    %cd {repo_dir}
else:
  print("Running locally - skipping git clone")

In [ ]:
# Import modules and packages
import os
import pandas as pd
import geopandas as gpd
from osgeo import gdal
pd.options.display.float_format = '{:.3f}'.format
gdal.UseExceptions()

import utils_ea_reprojection as ea
import utils_stratified_random_sampling as srs

In [ ]:
## Paths
base_path = os.getcwd() if in_colab() else '.'
input_data_path = os.path.join(base_path, 'input_data')
outputs_path = os.path.join(base_path, 'outputs')
os.makedirs(outputs_path, exist_ok=True)

# Each county's raw classification (EPSG:4326) and a short id_prefix used to keep
# draw_samples_nested's internal _sample_key unique across counties (both share the same
# stratum codes 0/1, so the prefix is what disambiguates them).
COUNTIES = {
    'bungoma': {
        'raster': os.path.join(input_data_path, 'BungomaCropland2025.tif'),
        'id_prefix': 'BGM25LG',
    },
    'muranga': {
        'raster': os.path.join(input_data_path, 'MurangaCropland2025.tif'),
        'id_prefix': 'MUR25LG',
    },
}

#### **Step 1.** Reproject each county's map to its own equal-area projection

>Pixel counting (hectares per stratum) and equal-probability stratified sampling both assume every pixel covers the same amount of ground area. In geographic coordinates (**EPSG:4326**) that's false: a degree of longitude covers less real ground distance near the poles than at the equator, so a fixed-size pixel in degrees does **not** represent a fixed amount of ground area across (or even within) the raster. Any hectare calculation or "uniformly sample a pixel" procedure done directly on it would be distorted -- unequally sized pixels get an unequal chance of selection, and area sums would be biased.
>
>Reprojecting to an **equal-area projection** removes this distortion: every pixel covers the same ground area everywhere in the raster, so pixel counts translate directly and unbiasedly into hectares, and every pixel gets an equal chance of being sampled.
>
>Rather than hardcoding a CRS, the cell below **auto-derives** a Lambert Azimuthal Equal-Area (LAEA) projection centered on each raster's own bounding-box centroid -- **separately per county**, since Bungoma and Muranga are different regions and a single shared projection centered on one would distort the other.

In [ ]:
ea_rasters = {}
for county, cfg in COUNTIES.items():
    # Nearest-neighbour resampling is required here: the raster is categorical (0/1 class
    # codes), and any other resampling method would blend/interpolate those codes into
    # meaningless values.
    proj_str = ea.derive_ea_proj_string(
        cfg['raster'], out_proj_path=os.path.join(outputs_path, f'{county}2025_ea_proj.txt')
    )
    print(f'{county}: auto-derived equal-area projection:\n  {proj_str}')

    ea_raster = os.path.join(outputs_path, os.path.basename(cfg['raster']).replace('.tif', '_ea.tif'))
    ea.raster_to_ea(cfg['raster'], ea_raster, proj_str, resampling_method='nearest')
    ea_rasters[county] = ea_raster

#### **Background** -- the statistically rigorous process, and why this notebook simplifies it

>`area_estimation_bungoma2025.ipynb` (and its script equivalent, `main.py`) implements the statistically rigorous version of this design, in two annotation rounds:
>
>1. **Pilot sample.** Draw a small, proportionally-allocated stratified random sample (`pilot_n` units) directly from the map -- proportional allocation is a neutral default here, since no per-stratum variance estimate exists yet.
>2. **Pilot annotation.** Interpret the pilot sample in STAC Notator to get its first reference labels (*round-1 checkpoint*).
>3. **Neyman priors (Sh).** From the annotated pilot's per-stratum user's accuracy, estimate a prior standard deviation `Sh = sqrt(Ui*(1-Ui))` for each stratum.
>4. **Neyman sample size (n_tot).** Combine `Sh` with a user-set target coefficient of variation (`cv_target`) and confidence level to compute the *total* sample size needed to hit that accuracy threshold: `n_tot = z^2 * (sum_h Wh*Sh)^2 / E^2`.
>5. **Neyman allocation (n_h).** Split `n_tot` across strata in proportion to each stratum's contribution to overall variance (`Wh*Sh`), so large, uncertain strata get proportionally more units -- this is what makes the design *optimal* for a given `n_tot`.
>6. **Full sample, nested with the pilot.** Draw the full `n_h`-per-stratum sample with the *same random seed* as the pilot, so the pilot's own sampled units -- and once annotated, their labels -- are reused rather than re-drawn or re-annotated.
>7. **Round-2 annotation.** Only the units NOT already covered by the pilot are exported for a second annotation round (*round-2 checkpoint*).
>8. **Combine both rounds** into one fully annotated sample.
>9. **Design-based area and accuracy estimates**, with uncertainty (Olofsson et al., 2014).
>
>That design is optimal because `n_tot` and its allocation are *derived from data* (the pilot's own observed variances) to hit a chosen accuracy target at the smallest possible sample size. The cost is two separate annotation rounds, with a pause between them while `Sh`/`n_tot`/allocation get computed -- workable for a single analyst or small team, but a poor fit for a **large training group**: pausing dozens of participants mid-session to recompute an allocation, then sending out a second, differently-sized batch, adds coordination overhead disproportionate to the training's purpose.
>
>**This notebook trades the statistically-optimal Neyman design for a single-round design sized to the group itself:**
>- No pilot round, no annotated-pilot variances, no Neyman sample-size formula or Neyman allocation.
>- The total sample size (`Ni_total`) is set directly from classroom capacity: `Ni_total = n_participants * n_samples_per_participant` -- and here, split evenly across counties.
>- Each county's share is allocated across its own strata **proportionally** to stratum area (`Wh`) -- the same neutral allocation the pilot itself uses above -- optionally with a **minimum per stratum** (`min_allocation`) so a small stratum still gets enough units for a meaningful accuracy check, rather than being rounded down to (near) zero.
>- One annotation round, one combined export, one combined import -- no nesting, no reconciliation, no combining two files.
>
>The resulting sample size is *not* tuned to hit a target CV the way the pilot + Neyman design's is -- it is whatever `n_participants * n_samples_per_participant` happens to be. Report the resulting precision (Step 7 below) as an outcome of the design actually used, not a pre-committed target.
>
>Everything below runs **independently per county** (own pixel counts, own allocation, own area/accuracy estimate), except the sample draw's export/annotation step, which is combined into a single file so participants only have one CSV to work with in STAC Notator.

#### **Step 2.** Strata, target stratum, and the shared sample-size parameters

>Strata are the same in both counties: **non-cropland** (0) and **cropland** (1).
>
>`n_participants` and `n_samples_per_participant` describe the training group as a whole -- the combined sample size (`Ni_total`, computed in Step 4) is `n_participants * n_samples_per_participant`, split evenly across counties. Participants aren't assigned to a specific county in this notebook: the combined sample is shuffled together before export, so anyone working through it in row order sees a mix of both counties' units. How the exported rows actually get distributed among participants is a logistics step handled outside this notebook.
>
>`min_allocation` (per county) is set in Step 4, after looking at each county's own pixel counts from Step 3 -- it isn't known yet at this point.
>
>`confidence` is still used to report each county's final area estimate precision (Step 7), even though it no longer drives the sample size.
>
>`seed` is reused for both counties' sample draws and the combined shuffle, for reproducibility.
>
>`id_col`/`true_col` and `STRATUM_TRUE_LABELS` mean the same as in the standard notebook's Step 2 -- they describe how the annotation tool codes classes in its own `true_col` output, so it can be relabeled onto the map's own coding before comparison. `annotated_path` is where this notebook looks for the single, combined annotated file once STAC Notator work is done.

In [ ]:
# Strata of interest
strata = [0, 1]
STRATUM_LABELS = {0: 'Non-cropland', 1: 'Cropland'}
target_stratum = 1  # cropland: the stratum the final area estimates highlight

# How the annotation tool codes the SAME classes in its own true-label output --
# see Step 2 of area_estimation_bungoma2025.ipynb for the full explanation.
STRATUM_TRUE_LABELS = {'Non-cropland': 'Non-cropland', 'Cropland': 'Cropland'}

# ---- User-defined parameters ----
n_participants = 20              # training group size (combined, not per county)
n_samples_per_participant = 40   # sample units per participant, across the combined pool of both counties
confidence = 0.95                # confidence level used to report each county's final area estimate CI
seed = 2025                      # random seed, reused for both counties' draws and the combined shuffle

# Column names expected in the annotation-tool export -- see Step 2 of
# area_estimation_bungoma2025.ipynb for details.
id_col = 'id'
true_col = 'stacnotator_label_name'

# Where to find the single, combined annotated file once STAC Notator work is done --
# update this path if it's saved somewhere other than outputs/.
annotated_path = os.path.join(outputs_path, 'large_groups_sample_annotated.csv')

#### **Step 3.** Pixel counts, per county

>Same as the standard workflow, run once per county: pixel counting gives each stratum's mapped area (`Area_ha`) and area weight (`Wi`), computed on that county's own equal-area raster from Step 1. Look at both counties' `Wi` here before setting `min_allocation` in Step 4.

In [ ]:
pixel_counts = {}
for county in COUNTIES:
    pixel_counts_csv = os.path.join(outputs_path, f'pixelcounts_{county}2025.csv')
    pixel_counts[county] = srs.compute_pixel_counts(ea_rasters[county], strata=strata, output_csv=pixel_counts_csv)
    print(f'--- {county} ---')
    display(pixel_counts[county])

#### **Step 4.** Sample size and allocation, per county

>The combined total sample size is `Ni_total = n_participants * n_samples_per_participant` (from Step 2), split evenly across counties: `Ni = Ni_total // len(COUNTIES)`.
>
>`min_allocation` is set **per county** here, after having looked at each county's own `Wi` in Step 3 above -- raise a county's value above 0 if a small stratum in that county would otherwise get very few (or zero) units under strict proportional allocation.
>
>Each county's `Ni` is then allocated across its own strata with the same `allocate_proportional` function the standard notebook uses for its pilot sample: each stratum gets `round(Ni * Wh)` units, at least that county's `min_allocation`.

In [ ]:
Ni_total = n_participants * n_samples_per_participant
Ni = {county: Ni_total // len(COUNTIES) for county in COUNTIES}
print(f'Ni_total = {n_participants} participants x {n_samples_per_participant} samples/participant = {Ni_total}, '
      f'split evenly per county: {Ni}')

# ---- User-defined parameter, set per county after looking at Step 3's pixel counts above ----
min_allocation = {
    'bungoma': 100,
    'muranga': 100,
}

allocation = {}
for county in COUNTIES:
    allocation_csv = os.path.join(outputs_path, f'large_groups_sample_allocation_{county}.csv')
    allocation[county] = srs.allocate_proportional(
        pixel_counts[county], n_total=Ni[county], min_allocation=min_allocation[county], output_csv=allocation_csv
    )
    print(f'{county} allocation:', allocation[county])

#### **Step 5.** Draw one sample per county, merge, shuffle once, export one combined file

>`draw_samples_nested` is called once per county, against that county's own raster and allocation, with a distinct `id_prefix` so the internal `_sample_key` (which encodes stratum and draw order) can't collide between counties sharing the same stratum codes.
>
>Each county's sample comes out in **that county's own** auto-derived equal-area CRS from Step 1 -- Bungoma's and Muranga's differ, since they're centered on different regions -- so each is reprojected to EPSG:4326 before the two are combined into one GeoDataFrame.
>
>The combined sample is then shuffled **once**, and given **one shared id space** (0..N-1) -- so the exported CSV mixes both counties' units in random order, the same way the single-county version shuffles before export.
>
>Before exporting, this id-to-county assignment is also saved to a small local lookup file. That's the notebook's own record for splitting the annotated results back out by county in Step 6 -- it does **not** rely on the `county` column that's also included in the exported CSV (for the interpreter's own context) surviving STAC Notator's annotation round-trip untouched, since that isn't guaranteed the way `id`/`true_col` already have to be.

In [ ]:
county_gdfs = []
for county, cfg in COUNTIES.items():
    gdf = srs.draw_samples_nested(ea_rasters[county], allocation[county], seed=seed, id_prefix=cfg['id_prefix'], v=True)
    gdf = gdf.to_crs(4326)  # each county's own EA CRS differs -- share one CRS before combining
    gdf['county'] = county
    county_gdfs.append(gdf)

combined_gdf = pd.concat(county_gdfs, ignore_index=True)
combined_gdf = gpd.GeoDataFrame(combined_gdf, geometry='geometry', crs='EPSG:4326')

# Shuffle the combined sample once, then assign one shared id space from that shuffled order --
# see shuffle_samples/assign_ids docstrings for why order matters here.
combined_gdf = srs.shuffle_samples(combined_gdf, seed=seed)
combined_gdf = srs.assign_ids(combined_gdf, id_col=id_col, v=True)

# Local, authoritative record of each id's county -- kept for Step 6, independent of
# whatever STAC Notator's own export does or doesn't preserve.
county_lookup_path = os.path.join(outputs_path, 'large_groups_sample_county_lookup.csv')
combined_gdf[[id_col, 'county']].to_csv(county_lookup_path, index=False)

# Export the combined sample (reprojected to EPSG:4326) for photo-interpretation in STAC Notator.
# `county` is included for the interpreter's own context but is not relied on downstream.
sample_csv, _ = srs.export_sample_units(
    combined_gdf,
    os.path.join(outputs_path, 'large_groups_sample_for_annotation.csv'),
    stratum_labels=STRATUM_LABELS, id_col=id_col, extra_cols=['county'], v=True,
)

> **Checkpoint -- annotation required.**
> Download `outputs/large_groups_sample_for_annotation.csv` -- one combined file covering both counties -- and interpret it in **STAC Notator** against the best available imagery for the 2025 season, recording each unit's *true* class. (How its rows get divided up among participants is a logistics step outside this notebook.) Export/combine the results as a single CSV with at least an `id` column (matching `id_col`) and a `true_col` column -- with `true_col` set to `'stacnotator_label_name'`, STAC Notator's own label-name export column can be used directly with no renaming. Save the combined file to `annotated_path` (set in Step 2), then run the next cell.

#### **Step 6.** Load the annotated sample, split by county

>The annotation tool's raw `true_col` values are relabeled onto the map's own stratum coding (via `STRATUM_TRUE_LABELS`/`STRATUM_LABELS`, same as the standard notebook) directly on the combined annotated file.
>
>Each row's county is then attached from the local lookup file written in Step 5 -- not from any `county` column that may or may not have come back from STAC Notator -- and used to split the combined annotated data into one CSV per county, so `load_full_annotations` (with its existing missing-column/unlabeled-row checks) can be reused per county for Steps 7-8.

In [ ]:
if not os.path.exists(annotated_path):
    raise FileNotFoundError(
        f"Annotated sample not found: {annotated_path}\n"
        "Annotate outputs/large_groups_sample_for_annotation.csv in STAC Notator first, "
        f"save the result with columns ['{id_col}', '{true_col}'] to this path, then re-run this cell."
    )

annotated_raw = pd.read_csv(annotated_path)
annotated_df = srs.relabel_true_stratum(annotated_raw, true_col, STRATUM_LABELS, STRATUM_TRUE_LABELS)

# County comes from the local lookup written in Step 5, not from `annotated_df` itself -- see
# Step 5's markdown for why that's not something this notebook trusts STAC Notator to
# preserve. Drop any `county` column that did come back, so there's no ambiguity about which
# one is authoritative.
county_lookup = pd.read_csv(county_lookup_path)[[id_col, 'county']]
if 'county' in annotated_df.columns:
    annotated_df = annotated_df.drop(columns=['county'])
annotated_df = annotated_df.merge(county_lookup, on=id_col, how='left')

unmatched = annotated_df['county'].isna()
if unmatched.any():
    ids = annotated_df.loc[unmatched, id_col].tolist()
    raise ValueError(
        f"{len(ids)} annotated id(s) not found in the county lookup ({county_lookup_path}): "
        f"{ids[:5]}{'...' if len(ids) > 5 else ''}"
    )

annotated_resolved_path = os.path.join(outputs_path, 'large_groups_sample_annotated_resolved.csv')
annotated_df.to_csv(annotated_resolved_path, index=False)

pred, true = {}, {}
for county in COUNTIES:
    county_resolved_path = os.path.join(outputs_path, f'large_groups_sample_annotated_resolved_{county}.csv')
    annotated_df[annotated_df['county'] == county].to_csv(county_resolved_path, index=False)
    pred[county], true[county], _ = srs.load_full_annotations(
        county_resolved_path, id_col=id_col, stratum_col='stratum', true_col=true_col
    )

annotated_df.head()

#### **Step 7.** Design-based area estimate, per county

>`compute_stratified_random_sampling_metrics` cross-tabulates each unit's map stratum (`pred`) against its annotated stratum (`true`) and turns the sample counts into an unbiased area estimate per class, with standard error and 95% CI, in hectares and as a percentage of the estimated area (Olofsson et al., 2014, Eqs. 8-10) -- run once per county, each against that county's own pixel counts.

In [ ]:
metrics = {}
for county in COUNTIES:
    metrics_csv = os.path.join(outputs_path, f'area_estimates_{county}2025_large_groups.csv')
    metrics[county] = srs.compute_stratified_random_sampling_metrics(
        pixel_counts[county], pred[county], true[county], output_csv=metrics_csv, v=True
    )
    print(f'--- {county} ---')
    display(metrics[county])

#### **Step 8.** Map accuracy, per county

>`compute_accuracy_metrics` computes overall accuracy plus each stratum's user's accuracy (Ui -- of the pixels the map calls this class, what fraction really are) and producer's accuracy (Pi -- of the pixels that really are this class, what fraction the map called it), each with SE and 95% CI (Olofsson et al., 2014, Eqs. 1-3, 5-7) -- run once per county, each against that county's own pixel counts.

In [ ]:
accuracy_metrics = {}
overall_accuracy = {}
for county in COUNTIES:
    accuracy_csv = os.path.join(outputs_path, f'accuracy_metrics_{county}2025_large_groups.csv')
    accuracy_metrics[county], overall_accuracy[county] = srs.compute_accuracy_metrics(
        pixel_counts[county], pred[county], true[county], output_csv=accuracy_csv, v=True
    )
    print(f'--- {county} ---')
    display(accuracy_metrics[county])

### **Result**

In [ ]:
print('=== 2025-season cropland area estimates, large-group sample design ===')
for county in COUNTIES:
    cropland = metrics[county].loc[target_stratum]
    print(f'--- {county.capitalize()} County ---')
    print(f"  Cropland area: {cropland['Area_ha']:.0f} ha +/- {cropland['CI_Ha']:.0f} ha "
          f"({cropland['CI%'] * 100:.1f}% relative precision at {int(confidence * 100)}% confidence; "
          f"from Ni={Ni[county]} units, not a pre-set CV target)")
    print(f"  Overall map accuracy: {overall_accuracy[county]['O']:.3f} +/- {overall_accuracy[county]['CI']:.3f}")

This is the final 2025-season cropland area estimate for Bungoma and Muranga Counties from the large-group sample design -- two independently-allocated stratified samples, drawn and annotated together in one combined round, estimated separately. For the statistically-optimal two-round pilot + Neyman design (currently Bungoma-only), see `area_estimation_bungoma2025.ipynb`.